In [19]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

In [20]:
train=pd.read_csv("train.csv")
test=pd.read_csv("test.csv")

In [21]:
train.info()
train.head()
train.describe()
train["survived"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 445 entries, 0 to 444
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        445 non-null    int64  
 1   survived  445 non-null    int64  
 2   pclass    445 non-null    int64  
 3   sex       445 non-null    object 
 4   age       360 non-null    float64
 5   sibsp     445 non-null    int64  
 6   parch     445 non-null    int64  
 7   fare      445 non-null    float64
 8   embarked  443 non-null    object 
dtypes: float64(2), int64(5), object(2)
memory usage: 31.4+ KB


survived
0    266
1    179
Name: count, dtype: int64

In [22]:
# 処理前
before = pd.concat(
    [train[["age", "embarked"]].isnull().sum().rename("train_処理前"),
     test[["age", "embarked"]].isnull().sum().rename("test_処理前")],
    axis=1
)
print(before)

# 欠損値処理
age_median = train["age"].median()
embarked_mode = train["embarked"].mode()[0]

for df in [train, test]:
    df["age"] = df["age"].fillna(age_median)
    df["embarked"] = df["embarked"].fillna(embarked_mode)

# 処理後
after = pd.concat(
    [train[["age", "embarked"]].isnull().sum().rename("train_処理後"),
     test[["age", "embarked"]].isnull().sum().rename("test_処理後")],
    axis=1
)
print(after)

          train_処理前  test_処理前
age              85        92
embarked          2         0
          train_処理後  test_処理後
age               0         0
embarked          0         0


In [23]:
features=["pclass","sex","age","sibsp","parch","fare","embarked"]

X_train=pd.get_dummies(train[features])
X_test=pd.get_dummies(test[features])

X_train,X_test=X_train.align(X_test,join="left",axis=1,fill_value=0)

y_train=train["survived"]

X_train.head()

,pclass,age,sibsp,parch,fare,sex_female,sex_male,embarked_C,embarked_Q,embarked_S
0,1,35.0,1,0,53.1000,True,False,False,False,True
1,3,35.0,0,0,8.0500,False,True,False,False,True
2,3,2.0,3,1,21.0750,False,True,False,False,True
3,2,14.0,1,0,30.0708,True,False,True,False,False
4,1,58.0,0,0,26.5500,True,False,False,False,True


In [24]:

model = RandomForestClassifier(n_estimators=300, max_depth=5, random_state=0)
model.fit(X_train, y_train)

print(f"train accuracy: {model.score(X_train, y_train):.3f}")

train accuracy: 0.863


In [25]:
proba = model.predict_proba(X_test)[:, 1]
proba[:5]  

array([0.11217769, 0.97101703, 0.61391929, 0.13516636, 0.36818991])

In [26]:
submit = pd.DataFrame({"id": test["id"], "proba": proba})
submit.to_csv("submit.csv", index=False, header=False)

In [27]:
submit.head()

,id,proba
0,0,0.112178
1,1,0.971017
2,2,0.613919
3,5,0.135166
4,6,0.368190
